In [10]:
import pandas as pd
import os

weather_data_input_folder = '../data'
station_data_input_file = 'station_data_cleaning/us_stations.csv'
final_weather_data_folder = 'final_weather_data'  # Define output folder

# if not os.path.exists(final_weather_data_folder):
#     print("hello")
#     os.umask(0)
#     os.chmod(final_weather_data_folder, 0o666)
#     os.makedirs(final_weather_data_folder, mode=0o777)

stations_df = pd.read_csv(station_data_input_file, index_col=0)

In [11]:
for year in range(2000, 2025):
        input_file = os.path.join(weather_data_input_folder, f'{year}_processed.csv')
        output_file = os.path.join(final_weather_data_folder, f'{year}_weather_data.csv')

        if os.path.exists(input_file):
            print(f"Processing file: {input_file}")

            # Load the weather data
            weather_df = pd.read_csv(input_file)
            
            # Filter to only stations starting with 'US'
            weather_df = weather_df[weather_df['Station ID'].str.startswith('US')]

            # Format date from yyyymmdd to yyyy-mm-dd
            weather_df['Date'] = pd.to_datetime(weather_df['Date'], format='%Y%m%d').dt.strftime('%Y-%m-%d')

            # Merge weather data with station data using station ID
            merged_df = weather_df.merge(stations_df, how='left', left_on='Station ID', right_index=True)
            
            # Select and reorder columns
            final_df = merged_df[['ELEVATION', 'LATITUDE', 'LONGITUDE', 'Date', 'TMAX (°C)', 'TMIN (°C)', 'STATE']]

            # Save the processed data to CSV
            final_df.to_csv(output_file, index=False)

            # Filter the data for temperatures over 56.7°C
            high_temp_df = final_df[final_df['TMAX (°C)'] > 56.7]

            # Display the filtered data
            high_temp_df